In [1]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [2]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [3]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [4]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

👉 加载主训练数据
  原始训练样本数: 7973
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条
  → 正在增强 Density 数据，共 787 条


[23:44:08] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[23:44:08] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[23:44:08] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[23:44:08] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[23:44:08] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[23:44:08] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[23:44:08] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[23:44:08] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[23:44:08] SMILES Parse 

cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081
👉 划分 train / validation / test
  划分结果: train=8064, val=1008, test=1009
Loaded: 8064 1008 1009


In [6]:
from train_stage3 import prepare_property_datasets, finetune_property

In [7]:
properties = ["Tg", "FFV", "Tc", "Density", "Rg"]
datasets = prepare_property_datasets(properties, train_df, val_df, test_df)

In [ ]:
tg_data = datasets["Tg"]
train_tg, val_tg, test_tg = tg_data["train"], tg_data["val"], tg_data["test"]
finetune_property(
    train_tg,
    val_tg,
    property_name="Tg",
    best_params_path=
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)